# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', '-')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their fields (@id references)
record_sets = []
print("Listing available Record Sets and their Fields:")
for record_set in metadata.record_sets:
    print(f"- RecordSet: {record_set.id} (name: {record_set.name})")
    record_sets.append(record_set.id)
    print("  Fields (by @id):")
    for field in record_set.fields:
        col_ids = [c.id for c in getattr(field, 'columns', [])]
        print(f"    - Field: {field.id}, Columns: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract data from the primary record set that contains the protein data.

In [ ]:
# Try to identify a main record set (by @id)
main_record_set_id = None
for record_set in metadata.record_sets:
    # Try to pick the first or with a suggestive 'protein' or 'data' in the name
    if main_record_set_id is None or 'protein' in record_set.name.lower() or 'data' in record_set.name.lower():
        main_record_set_id = record_set.id
# Use all record_sets found
all_record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id} with {df.shape[0]} records. Columns: {df.columns.tolist()}")
    else:
        print(f"RecordSet {record_set_id} has no records.")

# For demonstration, select the first available record set with data
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {record_set_id}:")
    print(dataframes[record_set_id].columns.tolist())
    dataframes[record_set_id].head()
else:
    print("No records could be loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA on the first record set with data
df = dataframes[record_set_id]

# Identify numeric fields (columns with number dtype)
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field = numeric_fields[0]  # Use the first numeric field for illustration
    print(f"Using numeric field for filtering and normalization: {numeric_field}")
else:
    print("No numeric fields found. Please inspect the DataFrame to choose a numeric field.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # Use mean as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by a categorical field, if available
    group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = None
    # Try to find a groupable field with a reasonable number of groups
    for col in group_fields:
        if df[col].nunique() > 1 and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("Cannot perform numeric EDA: no numeric field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()
else:
    print("No available numeric field for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset for mass spectrometry-derived extracellular vesicle protein data using the `mlcroissant` library. We demonstrated how to load all record sets using their `@id`, inspect fields and columns, filter and normalize numeric fields, and visualize the data. The dataset structure encouraged reproducibility and maintainability by referencing all entities via their Croissant `@id` fields.

For deeper analysis, you may select different record sets or fields and adapt the filtering/grouping/EDA sections accordingly.